# Blood Donor Eligibility — Logistic Regression (FIXED)

### What was wrong before
The original cleaned dataset included three engineered binary flags (`has_medical_condition`, `low_hemoglobin`, `low_weight`) that **directly encoded the rule used to generate the target label** in this synthetic dataset. The model did not need to learn anything — it simply read the flags. This caused 100% accuracy on both train and test, which is **data leakage**, not a good model.

### What this notebook does instead
Uses only **raw, continuous features** that represent real-world donor information a model would genuinely need to learn from.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# ============================================================
# 1. LOAD & PREPARE DATA FROM RAW SOURCE
# We rebuild features from raw data to avoid leakage
# ============================================================

candidate_paths = [
    Path("blood_donation.csv"),
    Path("../DATASET/blood_donation.csv"),
]
dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find blood_donation.csv")

df = pd.read_csv(dataset_path)
print(f"Loaded: {dataset_path}")
print("Raw shape:", df.shape)

In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING (Leakage-Free)
# ============================================================

# Date processing
REFERENCE_DATE = pd.Timestamp("2025-01-01")  # Fixed for reproducibility

df["Last_Donation_Date"] = df["Last_Donation_Date"].replace("Never", np.nan)
df["Last_Donation_Date"] = pd.to_datetime(df["Last_Donation_Date"], dayfirst=True, errors="coerce")
df["Registration_Date"] = pd.to_datetime(df["Registration_Date"], dayfirst=True, errors="coerce")

df["days_since_last_donation"] = (REFERENCE_DATE - df["Last_Donation_Date"]).dt.days
df["days_since_last_donation"] = df["days_since_last_donation"].fillna(
    (REFERENCE_DATE - df["Registration_Date"]).dt.days
)

# Gender-based donation interval rule
df["required_interval"] = df["Gender"].apply(lambda x: 90 if str(x).lower() == "male" else 120)
df["eligible_by_interval"] = (df["days_since_last_donation"] >= df["required_interval"]).astype(int)

# Gender encoding
df["Gender_Male"] = (df["Gender"].str.lower() == "male").astype(int)
df["Gender_Female"] = (df["Gender"].str.lower() == "female").astype(int)

# Target
df["Eligible_for_Donation_binary"] = df["Eligible_for_Donation"].map({"Yes": 1, "No": 0})

# ============================================================
# WHY WE EXCLUDE CERTAIN FEATURES:
#
# has_medical_condition -> directly encodes 100% of medical-condition-based ineligibility
# low_hemoglobin        -> derived threshold that perfectly separates Hgb<12.5 cases
# low_weight            -> derived threshold that perfectly separates Wt<50 cases
#
# These three flags TOGETHER with the target form a tautology in this synthetic dataset.
# A real-world model must learn from the raw continuous values.
# ============================================================

FEATURES = [
    "Hemoglobin_g_dL",        # raw continuous
    "Weight_kg",               # raw continuous
    "Age",                     # raw continuous
    "days_since_last_donation",# derived temporal
    "eligible_by_interval",    # interval rule (not a leaky flag — many eligible fail on other criteria)
    "Total_Donations",         # count
    "Gender_Male",             # encoded categorical
    "Gender_Female",           # encoded categorical
]

X = df[FEATURES]
y = df["Eligible_for_Donation_binary"]

print("Feature set shape:", X.shape)
print("\nTarget distribution (%):\n",
      (y.value_counts(normalize=True) * 100).round(2))

In [ ]:
# ============================================================
# 3. STRATIFIED TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Testing size: ", X_test.shape)

In [ ]:
# ============================================================
# 4. PIPELINE + HYPERPARAMETER SEARCH
# ============================================================

pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42)),
])

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 5.0, 10.0],
    "model__solver": ["lbfgs"],
    "model__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

In [ ]:
# ============================================================
# 5. EVALUATE — TRAIN VS TEST (OVERFITTING CHECK)
# ============================================================

train_pred  = best_model.predict(X_train)
test_pred   = best_model.predict(X_test)
train_proba = best_model.predict_proba(X_train)[:, 1]
test_proba  = best_model.predict_proba(X_test)[:, 1]

train_f1  = f1_score(y_train, train_pred)
test_f1   = f1_score(y_test, test_pred)
train_auc = roc_auc_score(y_train, train_proba)
test_auc  = roc_auc_score(y_test, test_proba)
overfit_gap = train_f1 - test_f1

print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Train F1:       {train_f1:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Train ROC-AUC:  {train_auc:.4f}")
print(f"Test ROC-AUC:   {test_auc:.4f}")
print(f"Overfitting Gap (Train F1 - Test F1): {overfit_gap:.4f}")

if overfit_gap > 0.05:
    print("⚠️  Warning: Noticeable overfitting. Consider stronger regularization.")
else:
    print("✅ Good: Overfitting gap is acceptable.")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))